# 502: Probability of No Further Warming Under Net-Zero Kyoto Emissions

This notebook calculates the probability that warming declines after net-zero Kyoto GHG emissions are achieved.

**Criteria for decline:**
- Peak warming occurs before 2100
- Delta warming (peak - 2100) > 0.01°C

In [7]:
import os
from pathlib import Path

import dotenv
import pandas as pd

dotenv.load_dotenv()

True

In [8]:
# Configuration
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])

# Guardrails for classifying decline
PEAK_YEAR_THRESHOLD = 2100  # Peak must occur before this year
DELTA_WARMING_THRESHOLD = 0.01  # Minimum decline in °C to count as decline

In [17]:
# Input metrics file - update this with the correct manifest ID
MANIFEST_ID = "2026-05-26_14-58"
METRICS_FILE = OUTPUT_FOLDER / f"500_gsat_metrics_{MANIFEST_ID}.csv"

## Load and Filter Data

In [18]:
# Load metrics
metrics_df = pd.read_csv(METRICS_FILE)
print(f"Total rows: {len(metrics_df)}")
metrics_df.head()

Total rows: 9600


,model,scenario,magicc_flag,run_id,gsat_peak,gsat_peak_year,gsat_2100,delta_warming
0,COFFEE 1.1,ENGAGE-NPi2020-400f-lowBECCS,ALL,0,1.967,2061,1.816,0.151
1,COFFEE 1.1,ENGAGE-NPi2020-400f-lowBECCS,ALL,1,3.211,2082,3.150,0.061
2,COFFEE 1.1,ENGAGE-NPi2020-400f-lowBECCS,ALL,2,2.149,2060,1.976,0.173
3,COFFEE 1.1,ENGAGE-NPi2020-400f-lowBECCS,ALL,3,1.663,2059,1.434,0.229
4,COFFEE 1.1,ENGAGE-NPi2020-400f-lowBECCS,ALL,4,2.040,2069,1.918,0.123


In [19]:
# Filter for NZKyoto scenarios only
nzkyoto_df = metrics_df[metrics_df["scenario"].str.contains("NZKyoto", na=False)].copy()
print(f"NZKyoto scenarios: {len(nzkyoto_df)} rows")
print(f"Unique scenarios: {nzkyoto_df['scenario'].nunique()}")

NZKyoto scenarios: 0 rows
Unique scenarios: 0


In [24]:
metrics_df["scenario"].unique()

array(['ENGAGE-NPi2020-400f-lowBECCS',
       'DeepElectrification-SSP2-HighRenewables-900',
       'LowEnergyDemand (1.3/IPCC-AR6)',
       'ENGAGE-NPi2020-400f-lowBECCS_NZCO2', 'SusDev-SDP-1000',
       'DeepElectrification-SSP2-HighRenewables-900_NZCO2',
       'LowEnergyDemand (1.3/IPCC-AR6)_NZCO2', 'SusDev-SDP-1000_NZCO2'],
      dtype=object)

In [20]:
# Filter for ALL magicc_flag (total warming, not just CO2 or GHG components)
nzkyoto_all_df = nzkyoto_df[nzkyoto_df["magicc_flag"] == "ALL"].copy()
print(f"NZKyoto ALL scenarios: {len(nzkyoto_all_df)} rows")

NZKyoto ALL scenarios: 0 rows


## Classify Ensemble Members

In [21]:
# Apply guardrails to classify each ensemble member as "decline" or not
nzkyoto_all_df["is_decline"] = (
    (nzkyoto_all_df["gsat_peak_year"] < PEAK_YEAR_THRESHOLD) &
    (nzkyoto_all_df["delta_warming"] > DELTA_WARMING_THRESHOLD)
)

print(f"Ensemble members classified as decline: {nzkyoto_all_df['is_decline'].sum()}")
print(f"Ensemble members not classified as decline: {(~nzkyoto_all_df['is_decline']).sum()}")

Ensemble members classified as decline: 0
Ensemble members not classified as decline: 0


## Calculate Probability Per Scenario

In [14]:
# Group by model and scenario, calculate probability
decline_prob = nzkyoto_all_df.groupby(["model", "scenario"]).agg(
    n_ensemble=pd.NamedAgg(column="run_id", aggfunc="count"),
    n_decline=pd.NamedAgg(column="is_decline", aggfunc="sum"),
).reset_index()

# Calculate probability
decline_prob["prob_decline"] = decline_prob["n_decline"] / decline_prob["n_ensemble"]
decline_prob["prob_decline"] = decline_prob["prob_decline"].round(3)

print(f"Calculated probabilities for {len(decline_prob)} scenarios")
decline_prob.head(10)

Calculated probabilities for 0 scenarios


,model,scenario,n_ensemble,n_decline,prob_decline


In [15]:
round(
    decline_prob['prob_decline'].describe(percentiles=[0.05, 0.34, 0.5, 0.67, 0.95]) * 100,
    2
)

count    0.0
mean     NaN
std      NaN
min      NaN
5%       NaN
34%      NaN
50%      NaN
67%      NaN
95%      NaN
max      NaN
Name: prob_decline, dtype: float64

## Save Results

In [16]:
# Save the decline probability results
output_path = OUTPUT_FOLDER / f"501_nzkyoto_decline_probability_{MANIFEST_ID}.csv"
decline_prob.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

Results saved to: /Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output/501_nzkyoto_decline_probability_2026-05-26_14-58.csv
